In [1]:
# County Attributes — NLP + K-Means Clustering

## 1. Import libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
```

## 2. Load the dataset

```python
# Load county-attr.csv
df = pd.read_csv("county-attr.csv")

# Display the first 5 rows
display(df.head())

# Dataset information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())
```

## 3. Identify the text column

First inspect the columns and select the column containing the county description/text.

```python
# Display data types
print(df.dtypes)
```

For example, if the text column is called `description`:

```python
text_column = "description"

# Convert missing values to empty strings
df[text_column] = df[text_column].fillna("").astype(str)

# Preview text
display(df[[text_column]].head())
```

If your column has a different name, replace `"description"` with the actual column name.

## 4. Basic NLP text cleaning

```python
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # Remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)        # Keep letters and spaces
    text = re.sub(r"\s+", " ", text).strip()     # Remove extra spaces
    return text

df["clean_text"] = df[text_column].apply(clean_text)

display(df[[text_column, "clean_text"]].head())
```

## 5. Convert text into numerical features using TF-IDF

```python
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)
```

## 6. Find a suitable number of K-Means clusters

Use the Elbow Method.

```python
inertias = []

K = range(2, 11)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K, inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()
```

## 7. Calculate silhouette scores

```python
silhouette_scores = []

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X)
    
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(K, silhouette_scores, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for Different K Values")
plt.show()

for k, score in zip(K, silhouette_scores):
    print(f"K = {k}: Silhouette Score = {score:.4f}")
```

## 8. Run K-Means

Choose the number of clusters based on the graphs above.

For example:

```python
n_clusters = 4

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

display(df.head())
```

## 9. Count counties in each cluster

```python
cluster_counts = df["cluster"].value_counts().sort_index()

print(cluster_counts)
```

Visualise the result:

```python
plt.figure(figsize=(8, 5))

cluster_counts.plot(kind="bar")

plt.xlabel("Cluster")
plt.ylabel("Number of counties")
plt.title("Number of Counties in Each Cluster")
plt.xticks(rotation=0)
plt.show()
```

## 10. Find the important words in each cluster

```python
terms = tfidf.get_feature_names_out()
centers = kmeans.cluster_centers_

for cluster_num in range(n_clusters):
    top_indices = centers[cluster_num].argsort()[-15:][::-1]
    
    print(f"\nCluster {cluster_num}")
    print("-" * 40)
    
    for index in top_indices:
        print(terms[index])
```

## 11. Display counties belonging to each cluster

If your county-name column is called `county`:

```python
for cluster_num in range(n_clusters):
    print(f"\n===== CLUSTER {cluster_num} =====")
    
    cluster_data = df[df["cluster"] == cluster_num]
    
    display(cluster_data[["county", text_column]].head(20))
```

If your county column has a different name, replace `"county"` with the correct column name.

## 12. Create a cluster summary

```python
cluster_summary = []

for cluster_num in range(n_clusters):
    cluster_data = df[df["cluster"] == cluster_num]
    
    top_indices = centers[cluster_num].argsort()[-10:][::-1]
    top_words = [terms[i] for i in top_indices]
    
    cluster_summary.append({
        "Cluster": cluster_num,
        "Number_of_Counties": len(cluster_data),
        "Top_Words": ", ".join(top_words)
    })

summary_df = pd.DataFrame(cluster_summary)

display(summary_df)
```

## 13. Visualise the clusters

Because TF-IDF can contain thousands of dimensions, reduce it to two dimensions for plotting.

```python
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)

X_2d = svd.fit_transform(X)

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.7
)

plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("County Clusters Using NLP + K-Means")

plt.colorbar(scatter, label="Cluster")

plt.show()
```

## 14. Save the results

```python
df.to_csv("county-attr-clustered.csv", index=False)

summary_df.to_csv("county-cluster-summary.csv", index=False)

print("Clustered dataset saved as county-attr-clustered.csv")
print("Cluster summary saved as county-cluster-summary.csv")
```

## 15. Final result

The final dataset will contain the original county information plus:

* `clean_text` — cleaned NLP text
* `cluster` — K-Means cluster assigned to each county

For example:

```python
display(
    df[["county", text_column, "cluster"]].head(20)
)
```

### Complete workflow

The notebook follows this pipeline:

CSV
→ Load data
→ Clean text
→ TF-IDF
→ Determine K
→ K-Means
→ Analyse clusters
→ Visualise clusters
→ Save results

```
```


SyntaxError: invalid character '—' (U+2014) (1272276931.py, line 280)